# Análise SSP-DF × delegacias

Cruza as ocorrências por Região Administrativa com a localização das delegacias e monta os dois mapas. Toda a lógica vive em `src/tcc`; para gerar os HTML sem abrir o notebook use `scripts/gerar_mapas.py`.

In [ ]:
from tcc.geo import (
    carregar_setores,
    centroides_por_ra,
    delegacias_com_subdistrito,
    dissolver_por_ra,
    juntar_ssp_delegacias,
)
from tcc.mapas import mapa_bolhas, mapa_coropletico, preparar_dados
from tcc.paths import OUTPUT_DIR
from tcc.ssp_df import carregar_dados_ssp_df

## 1. Carregar SSP-DF

In [ ]:
ssp = carregar_dados_ssp_df()
print(f'SSP shape: {ssp.shape}')
ssp.head()

## 2. Setores censitários e delegacias (spatial join)

In [ ]:
gdf = carregar_setores()
gdf_ra = dissolver_por_ra(gdf)
dp_joined = delegacias_com_subdistrito(gdf)

print(f'Delegacias: {len(dp_joined)}')
dp_joined[['Delegacia', 'Região Administrativa', 'CD_SUBDIST', 'NM_SUBDIST']].head(10)

## 3. Agregar delegacias por RA e juntar no SSP

In [ ]:
ssp_dp = juntar_ssp_delegacias(ssp, dp_joined)

print(f'Shape final: {ssp_dp.shape}')
print(f"Nulls em n_delegacias: {ssp_dp['n_delegacias'].null_count()}")
ssp_dp.head()

## 4. Mapa coroplético: crimes por RA + delegacias

In [ ]:
dados = preparar_dados(ssp_dp)

mapa = mapa_coropletico(gdf, gdf_ra, dp_joined, dados)
mapa.save(str(OUTPUT_DIR / 'mapa_ssp_delegacias.html'))
mapa

## 5. Mapa de bolhas: total por RA no centroide

In [ ]:
mapa2 = mapa_bolhas(gdf, gdf_ra, dp_joined, dados, centroides_por_ra(gdf_ra))
mapa2.save(str(OUTPUT_DIR / 'mapa_bolhas_delegacias.html'))
mapa2